# Forecast Sell-In — com tratamento de picos (feriados + peso nos extremos)

Versão 2 do pipeline: além dos passos anteriores (agregação dos 3 canais, features da Table 4,
hold-out de 23 semanas, forecast recursivo), esta versão ataca diretamente a dificuldade do
case em prever os **picos** de demanda:

1. **Feature de feriado calculada por fórmula** (Carnaval, Páscoa, Corpus Christi, Dia das Mães
   etc. via algoritmo de Páscoa/Computus) — resolve o bloqueio de não ter uma base de feriados,
   já que 50-69% dos picos históricos ocorrem a ≤7 dias de um feriado (vs. 31% nas semanas normais).
2. **Peso maior nas amostras de pico durante o treino**, porque o gradient boosting com erro
   quadrático tende a *subestimar* sistematicamente os valores extremos (regressão à média).

> Pré-requisito: `pip install lightgbm pandas numpy matplotlib openpyxl scikit-learn python-dateutil`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime as dt
from dateutil.easter import easter
from lightgbm import LGBMRegressor

pd.set_option('display.max_columns', None)


## 1. Configuração

In [ ]:
SRC = 'Business_Case_Data_Set.xlsx'   # <-- ajuste o caminho no seu ambiente

PRODS = ['Energy Drink 350ml', 'Flavored Water 500ml', 'Natural Juice 1L']
HOLDOUT_WEEKS = 23

FEATURES = ['Lag_1w', 'Lag_4w', 'Rolling_Mean_4w', 'Rolling_Std_4w',
            'Price_per_kg_USD', 'Price_Change_Pct', 'Numeric_Distribution',
            'Weighted_Distribution', 'Advertising_Investment_USD',
            'Promotion_Investment_USD', 'Promo_Flag', 'Month_Sin', 'Month_Cos',
            'Interaction_Price_Promo',
            'Days_To_Holiday', 'Near_Holiday_Flag']   # <- novas features de feriado
TARGET = 'Sell_In_Tons_Total'

# Parâmetros da Table 3 - Model Parameters (LGBM)
MODEL_PARAMS = dict(
    n_estimators=500, learning_rate=0.1, max_depth=12, num_leaves=256,
    min_child_samples=5, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=0.0, reg_lambda=0.0, min_split_gain=0.0,
    boosting_type='gbdt', objective='regression', metric='mape',
    random_state=42, n_jobs=-1, verbose=-1,
)

PEAK_QUANTILE = 0.75   # define o que é "semana de pico" no treino
PEAK_WEIGHT = 3.0      # peso extra dado a essas semanas no fit


## 2. Calendário de feriados móveis/fixos (calculado, sem base externa)

Datas móveis calculadas via algoritmo de Páscoa (Gauss/Computus, `dateutil.easter`).
Isso resolve o bloqueio original: não temos uma planilha de feriados, mas dá para
**calcular** as datas relevantes para o mercado de bebidas no Brasil.


In [ ]:
def second_sunday_may(y):
    d = dt.date(y, 5, 1)
    sundays = [d + dt.timedelta(days=i) for i in range(31)
               if (d + dt.timedelta(days=i)).month == 5 and (d + dt.timedelta(days=i)).weekday() == 6]
    return sundays[1]

def brazil_holiday_dates(years):
    hols = []
    for y in years:
        e = easter(y)
        hols += [e - dt.timedelta(days=47),   # Carnaval
                 e,                             # Páscoa
                 second_sunday_may(y),          # Dia das Mães
                 e + dt.timedelta(days=60),     # Corpus Christi
                 dt.date(y, 6, 12),             # Dia dos Namorados
                 dt.date(y, 6, 24),             # São João / Festa Junina
                 dt.date(y, 10, 12),            # Dia das Crianças
                 dt.date(y, 12, 25),            # Natal
                 dt.date(y, 1, 1)]              # Ano Novo
    return sorted(set(hols))

HOL_DATES = brazil_holiday_dates(range(2022, 2029))

def dist_to_holiday(week_date):
    wd = week_date.date() if hasattr(week_date, 'date') else week_date
    return min(abs((wd - h).days) for h in HOL_DATES)


## 3. Carga, agregação (soma dos 3 canais) e engenharia de features

In [ ]:
def wavg(group, valcol, wcol):
    w = group[wcol]
    return (group[valcol] * w).sum() / w.sum() if w.sum() != 0 else group[valcol].mean()


def load_and_aggregate():
    df1 = pd.read_excel(SRC, sheet_name='Table 1 - External Variables',
                         keep_default_na=False, na_values=[''])
    df2 = pd.read_excel(SRC, sheet_name='Table 2 - Sell In')
    d2 = df2[df2['Product'].isin(PRODS)].copy()
    d1 = df1[df1['Product'].isin(PRODS)].copy()
    d1['Week'], d2['Week'] = pd.to_datetime(d1['Week']), pd.to_datetime(d2['Week'])
    m = d2.merge(d1, on=['Week', 'Product', 'Channel'], how='left')
    m['Promo_Flag_ch'] = (m['Promotion_Type'] != 'None').astype(int)

    recs = []
    for (prod, week), g in m.groupby(['Product', 'Week']):
        recs.append({
            'Product': prod, 'Week': week,
            'Sell_In_Tons_Total': g['Sell_In_Tons'].sum(),
            'Price_per_kg_USD': wavg(g, 'Price_per_kg_USD', 'Sell_In_Tons'),
            'Numeric_Distribution': wavg(g, 'Numeric_Distribution', 'Sell_In_Tons'),
            'Weighted_Distribution': wavg(g, 'Weighted_Distribution', 'Sell_In_Tons'),
            'Advertising_Investment_USD': g['Advertising_Investment_USD'].sum(),
            'Promotion_Investment_USD': g['Promotion_Investment_USD'].sum(),
            'Promo_Flag': 1 if g['Promo_Flag_ch'].max() > 0 else 0,
        })
    return pd.DataFrame(recs).sort_values(['Product', 'Week']).reset_index(drop=True)


def engineer_features(agg):
    feat_rows = []
    for prod, g in agg.groupby('Product'):
        g = g.sort_values('Week').reset_index(drop=True)
        s = g['Sell_In_Tons_Total']
        g['Lag_1w'] = s.shift(1)
        g['Lag_4w'] = s.shift(4)
        g['Rolling_Mean_4w'] = s.shift(1).rolling(4).mean()   # t-4..t-1
        g['Rolling_Std_4w'] = s.shift(1).rolling(4).std()     # t-4..t-1
        g['Price_Change_Pct'] = g['Price_per_kg_USD'].pct_change()
        g['Month_Sin'] = np.sin(2 * np.pi * g['Week'].dt.month / 12)
        g['Month_Cos'] = np.cos(2 * np.pi * g['Week'].dt.month / 12)
        g['Interaction_Price_Promo'] = g['Price_per_kg_USD'] * g['Promo_Flag']
        g['Days_To_Holiday'] = g['Week'].apply(dist_to_holiday)
        g['Near_Holiday_Flag'] = (g['Days_To_Holiday'] <= 7).astype(int)
        feat_rows.append(g)
    return pd.concat(feat_rows, ignore_index=True)


agg = load_and_aggregate()
feat = engineer_features(agg)
feat.tail()


## 4. Diagnóstico rápido: os picos têm relação com feriado?

Compara a % de semanas de pico (top 10% de volume) que caem perto de um feriado
vs. a % geral — e a distância média até o feriado mais próximo.


In [ ]:
for prod, g in feat.groupby('Product'):
    g = g.sort_values('Week')
    thresh = g[TARGET].quantile(0.90)
    spikes = g[g[TARGET] >= thresh]
    nonspikes = g[g[TARGET] < thresh]
    pct_spike = (spikes['Days_To_Holiday'] <= 7).mean() * 100
    pct_all = (g['Days_To_Holiday'] <= 7).mean() * 100
    print(f"{prod:22s}  picos perto de feriado: {pct_spike:5.1f}%   "
          f"(baseline geral: {pct_all:5.1f}%)   "
          f"dist. média pico={spikes['Days_To_Holiday'].mean():.1f}d  "
          f"outras={nonspikes['Days_To_Holiday'].mean():.1f}d")


## 5. Treino + forecast recursivo, com peso extra nas semanas de pico

O `sample_weight` dá peso 3x maior às semanas de treino que estão no top 25% de volume —
isso combate o viés de subestimação de picos típico de modelos treinados com erro quadrático.


In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))


def train_and_forecast(feat, use_peak_weight=True):
    results, plot_data, models = {}, {}, {}

    for prod in PRODS:
        g = feat[feat['Product'] == prod].sort_values('Week').reset_index(drop=True)
        n = len(g)
        train_end = n - HOLDOUT_WEEKS

        train = g.iloc[:train_end].dropna(subset=FEATURES + [TARGET])
        sample_weight = None
        if use_peak_weight:
            p = train[TARGET].quantile(PEAK_QUANTILE)
            sample_weight = np.where(train[TARGET] >= p, PEAK_WEIGHT, 1.0)

        model = LGBMRegressor(**MODEL_PARAMS)
        model.fit(train[FEATURES], train[TARGET], sample_weight=sample_weight)
        models[prod] = model

        working = g[TARGET].tolist()
        preds = []
        for t in range(train_end, n):
            lag1, lag4 = working[t - 1], working[t - 4]
            roll_window = working[t - 4:t]
            roll_mean, roll_std = np.mean(roll_window), np.std(roll_window, ddof=1)
            row = g.iloc[t]
            x = pd.DataFrame([{
                'Lag_1w': lag1, 'Lag_4w': lag4,
                'Rolling_Mean_4w': roll_mean, 'Rolling_Std_4w': roll_std,
                'Price_per_kg_USD': row['Price_per_kg_USD'],
                'Price_Change_Pct': row['Price_Change_Pct'],
                'Numeric_Distribution': row['Numeric_Distribution'],
                'Weighted_Distribution': row['Weighted_Distribution'],
                'Advertising_Investment_USD': row['Advertising_Investment_USD'],
                'Promotion_Investment_USD': row['Promotion_Investment_USD'],
                'Promo_Flag': row['Promo_Flag'],
                'Month_Sin': row['Month_Sin'], 'Month_Cos': row['Month_Cos'],
                'Interaction_Price_Promo': row['Interaction_Price_Promo'],
                'Days_To_Holiday': row['Days_To_Holiday'],
                'Near_Holiday_Flag': row['Near_Holiday_Flag'],
            }])[FEATURES]

            yhat = max(model.predict(x)[0], 0)
            preds.append(yhat)
            working[t] = yhat  # realimenta a própria previsão (recursivo)

        actual = g[TARGET].iloc[train_end:n].values
        weeks = g['Week'].iloc[train_end:n].values
        m_ = mape(actual, preds)
        results[prod] = {'MAPE': m_, 'Accuracy_1_minus_MAPE': 1 - m_,
                          'n_train': len(train), 'n_holdout': len(preds)}
        plot_data[prod] = pd.DataFrame({'Week': weeks, 'Actual': actual, 'Predicted': preds})

    return pd.DataFrame(results).T.astype(float), plot_data, models


res_df, plot_data, models = train_and_forecast(feat, use_peak_weight=True)
res_df


## 6. Desempenho específico nas semanas de pico do hold-out

In [ ]:
print("MAPE e viés restritos às semanas de pico (top 25% do Actual no hold-out):\n")
for prod in PRODS:
    d = plot_data[prod]
    thresh = d['Actual'].quantile(0.75)
    mask = d['Actual'] >= thresh
    peak_mape = mape(d.loc[mask, 'Actual'], d.loc[mask, 'Predicted'])
    bias = (d.loc[mask, 'Actual'] - d.loc[mask, 'Predicted']).mean()
    print(f"{prod:22s}  n_picos={mask.sum():2d}  MAPE_pico={peak_mape*100:5.1f}%  "
          f"viés médio={bias:+.2f}t {'(subestima)' if bias>0 else '(superestima)'}")


## 7. Gráfico — Real vs Previsto (hold-out), com picos destacados

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 13))
for ax, prod in zip(axes, PRODS):
    d = plot_data[prod]
    ax.plot(d['Week'], d['Actual'], marker='o', color='#2E5C8A', label='Real', linewidth=2.2, zorder=3)
    ax.plot(d['Week'], d['Predicted'], marker='s', color='#D9534F', label='Previsto (recursivo)',
            linewidth=2, linestyle='--')
    thresh = d['Actual'].quantile(0.75)
    peaks = d[d['Actual'] >= thresh]
    ax.scatter(peaks['Week'], peaks['Actual'], color='#F0AD4E', s=110, zorder=4,
               marker='*', label='Semana de pico')

    a = res_df.loc[prod]
    ax.set_title(f"{prod} — Hold-out (últimas {HOLDOUT_WEEKS} semanas)  |  "
                 f"MAPE: {a['MAPE']*100:.1f}%  |  Accuracy: {a['Accuracy_1_minus_MAPE']*100:.1f}%",
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Sell-In (Tons)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')
plt.tight_layout()
plt.savefig('holdout_forecast_final.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Próximos passos possíveis (não implementados aqui)

- **Regressão quantílica** (`objective='quantile'` no LightGBM, alpha=0.1/0.5/0.9): em vez de um
  único ponto, gerar um intervalo de previsão (P10-P90) — mais honesto para o planejamento de
  estoque em produtos com picos difíceis de acertar exatamente.
- **Modelo em duas etapas**: um classificador binário "esta semana vai ser pico?" (usando
  `Near_Holiday_Flag`, sazonalidade, histórico) + um regressor condicional — separa o problema de
  *quando* vai ter pico do problema de *quanto* vai vender.
- **Mais anos de histórico** ao redor de cada feriado ajudariam o modelo a aprender o padrão com
  mais confiança (hoje temos só 3 ocorrências de cada feriado no dataset).
